# 03 · Feature Engineering
### BNI Ecosystem & Growth Intelligence — Kelompok 8 · ODP BNI Data Analytics Batch 2

| # | Notebook | Isi |
|---|---|---|
| 01 | Dataset Generation & Validation | data sintetis v8 + validasi |
| 02 | EDA | temuan, hipotesis, kontrak fitur |
| **03** | **Feature Engineering** | **notebook ini — split train/valid/test, dataset fitur final, uji nilai fitur** |
| 04 | Modelling | baseline risiko & anomali |
| 05 | Modelling Optimization | HPO, cost-sensitive, kalibrasi |
| 06 | Model Evaluation | evaluasi di test & ground truth |
| 07 | Post-evaluation & Remodelling | perbaikan dari analisis kesalahan |

## Tujuan
Notebook ini mengubah 27 CSV mentah menjadi **dua tabel fitur final**, sehingga notebook 04–07 dan dashboard tidak perlu join banyak CSV lagi:

| Tabel | Butir | Untuk |
|---|---|---|
| `features_risk` | 1 baris per perusahaan (2.000) | klasifikasi risiko |
| `features_anomaly_panel` | 1 baris per nasabah × bulan (1.400 × 30) | deteksi anomali "Why Now" |

Setiap fitur berasal dari **kontrak fitur** notebook 02 (`outputs/02_eda/feature_contract.csv`) dan diberi **tingkat bukti**:
- **A** — sudah diuji di EDA;
- **B** — turunan dari temuan EDA, bentuknya diuji di notebook ini;
- **C** — praktik umum / pelengkap, hanya dipertahankan kalau lolos ablation.

## Prinsip yang dijaga
1. **Split train / valid / test untuk HPO.** Test = holdout notebook 02 (tidak berubah, tidak pernah dilihat). Dev dipecah menjadi train dan valid. Disediakan juga `cv_fold` untuk HPO berbasis cross-validation. Alasannya: memilih hiperparameter dengan data yang sama dengan data penilaian akhir membuat performa terlihat lebih baik dari kenyataan (Varma & Simon, 2006; Cawley & Talbot, 2010).
2. **Statistik di-fit tanpa valid & test.** Statistik peer (median/MAD) dan faktor musiman hanya dihitung dari train + perusahaan tanpa label. Untuk HPO berbasis CV, `src/features.py` menyediakan `PeerZScaler` yang di-fit ulang di setiap fold.
3. **Fitur anomali bersifat online.** Nilai di bulan *t* hanya memakai data sampai bulan *t*. Laporan keuangan baru dianggap tersedia 4 bulan setelah tahun buku berakhir.
4. **Pagar kolom.** Hanya kolom `peran = fitur`. Proxy-label, `gt_*`, dan outcome tidak dipakai. `relationship_end` tidak dipakai di fitur anomali (temuan F10: penanda langsung buyer exit). `data_visibility` dan volume transaksi tidak masuk set portabel prospect (G2).

## Input & output (relatif ke akar proyek)
| | Path | Asal / pemakai |
|---|---|---|
| **input** | `data/raw/*.csv` | output notebook 01 (data sintetis v8) |
| **input** | `data/interim/split.csv` | holdout test notebook 02 (`dev` / `holdout`) |
| **input** (opsional) | `data/interim/transactions_clean.(parquet\|csv.gz)` | transaksi bersih notebook 02; bila tidak ada, dibersihkan ulang dari `data/raw/transactions.csv` |
| **input** | `src/features.py` | `PeerZScaler`, `RATIO_COLS`, `PEER_KEYS` |
| **output** | `data/processed/features_risk.csv.gz`, `features_anomaly_panel.csv.gz` | dataset fitur final untuk notebook 04–07 |
| **output** | `data/processed/split.csv`, `feature_sets.json`, `feature_dictionary.csv` | split train/valid/test + `cv_fold`, daftar set fitur, kamus fitur |
| **output** | `data/processed/eval_targets/` | ground truth yang dirapikan (hanya untuk evaluasi) |
| **output** | `outputs/03_fe/*.csv`, `REFERENCES.md`, `figures/*.png` | hasil uji nilai fitur & rujukan |

Notebook ini mencari akar proyek (folder berisi `pyproject.toml`), jadi boleh dijalankan dari folder `notebooks/`. Waktu jalan ±3–5 menit (sebagian besar untuk ablation).

**Daftar isi:** A. Setup · B. Split · C. Fitur risiko · D. Fitur panel anomali · E. Uji nilai fitur · F. Simpan dataset final · Daftar pustaka

## A. Setup

In [ ]:
import json, re, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Markdown
from scipy import stats
from threadpoolctl import threadpool_limits
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 70)

SEED = 2026
CUTOFF = pd.Timestamp('2026-06-30')
REPORT_LAG_MONTHS = 4          # laporan tahun buku Y dianggap tersedia mulai April Y+1

In [ ]:
def find_root(start=Path.cwd()):
    """Akar proyek = folder terdekat (dari cwd ke atas) yang berisi pyproject.toml."""
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists(): return p
    raise FileNotFoundError('pyproject.toml tidak ditemukan — jalankan notebook dari dalam folder proyek')

PROJECT_ROOT = find_root()
SRC     = PROJECT_ROOT / 'src'
RAW     = PROJECT_ROOT / 'data' / 'raw'               # INPUT: output notebook 01
INTERIM = PROJECT_ROOT / 'data' / 'interim'           # INPUT: output notebook 02 (split holdout, transaksi bersih)
PROC    = PROJECT_ROOT / 'data' / 'processed'         # OUTPUT: dataset fitur final
EVAL    = PROC / 'eval_targets'                       # OUTPUT: ground truth, hanya untuk evaluasi
OUT     = PROJECT_ROOT / 'outputs' / '03_fe'          # OUTPUT: hasil uji nilai fitur
FIG     = OUT / 'figures'

REQUIRED = [RAW / 'companies.csv', RAW / 'transactions.csv', INTERIM / 'split.csv', SRC / 'features.py']
missing = [str(p.relative_to(PROJECT_ROOT)) for p in REQUIRED if not p.exists()]
if missing:
    raise FileNotFoundError('Input belum ada: ' + ', '.join(missing) + ' → lihat README (data/raw dari notebook 01, data/interim dari notebook 02).')
for d in (PROC, EVAL, OUT, FIG): d.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
from features import PeerZScaler, RATIO_COLS, PEER_KEYS

print('PROJECT_ROOT:', PROJECT_ROOT); print('data mentah :', RAW); print('output      :', PROC, '|', OUT)

In [ ]:
C = dict(s1='#2a78d6', s2='#eb6834', s3='#1baf7a', s4='#eda100', s5='#e87ba4', s6='#008300', s7='#6250d6', s8='#e34948',
         ink='#0b0b0b', ink2='#52514e', muted='#898781', grid='#e1e0d9', axis='#c3c2b7', surface='#fcfcfb')
RISK_ORDER = ['Low', 'Medium', 'High']
mpl.rcParams.update({'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'axes.edgecolor': C['axis'], 'axes.linewidth': .8,
                     'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': .6, 'axes.axisbelow': True, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'axes.labelcolor': C['ink2'], 'axes.labelsize': 9, 'xtick.color': C['muted'], 'ytick.color': C['muted'],
                     'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.frameon': False, 'legend.fontsize': 8, 'lines.linewidth': 2,
                     'figure.dpi': 110, 'text.color': C['ink']})
def save_fig(fig, name): fig.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())

def write_table(df, name, folder=PROC):
    """Simpan parquet bila pyarrow tersedia, kalau tidak csv.gz (src/features.read_table membaca keduanya)."""
    try:
        df.to_parquet(folder / f'{name}.parquet', index=False); return f'{name}.parquet'
    except Exception:
        df.to_csv(folder / f'{name}.csv.gz', index=False, compression='gzip'); return f'{name}.csv.gz'

In [ ]:
rd = lambda t, **k: pd.read_csv(RAW / f'{t}.csv', **k)
co = rd('companies'); fin = rd('financials'); rl = rd('risk_labels'); geo = rd('geography'); br = rd('branches'); ne = rd('network_entities')
cf = rd('credit_facilities'); cf = cf.drop(columns=[c for c in ['collectibility', 'max_dpd_12m', 'restructuring_flag', 'watchlist_flag'] if c in cf])
rel = rd('relationships', parse_dates=['relationship_start', 'relationship_end'])

def load_transactions():
    """Transaksi bersih dari notebook 02; kalau belum ada, bersihkan ulang dengan aturan D1 yang sama."""
    for f in ['transactions_clean.parquet', 'transactions_clean.csv.gz']:
        if (INTERIM / f).exists():
            t = pd.read_parquet(INTERIM / f) if f.endswith('parquet') else pd.read_csv(INTERIM / f)
            t['transaction_date'] = pd.to_datetime(t.transaction_date); return t, f
    t = rd('transactions', parse_dates=['transaction_date'])
    t = t[~t.transaction_id.duplicated()]; t = t[(t.amount_idr_mn > 0) & (t.transaction_date <= CUTOFF)]
    t['channel'] = t.channel.fillna('Tidak tercatat'); return t, 'raw (dibersihkan ulang)'
t0 = time.time(); tx, src_tx = load_transactions()
tx['ym'] = tx.transaction_date.dt.to_period('M'); tx['day'] = tx.transaction_date.dt.day
MONTHS = pd.period_range('2024-01', '2026-06', freq='M'); T = len(MONTHS)
print(f'transaksi: {len(tx):,} baris dari {src_tx} ({time.time() - t0:.1f} dtk) · {T} bulan')
CLIENTS = co.loc[co.customer_type != 'Prospect', 'company_id'].tolist()
IS_CLIENT = co.set_index('company_id').customer_type.ne('Prospect')

## B. Split train / valid / test (+ fold CV)
- **test** = holdout notebook 02 (200 nasabah berlabel). Tidak berubah dan belum pernah dipakai untuk apa pun.
- **dev** notebook 02 (800) dipecah menjadi **train (±600)** dan **valid (±200)**, bertingkat menurut kelas risiko dan per grup usaha (anggota grup tidak terbelah).
- **`cv_fold`** (0–4) diberikan ke seluruh baris train+valid, untuk HPO berbasis cross-validation.
- **unlabeled** = 1.000 nasabah non-kredit & prospect (untuk skoring dan statistik tanpa label).

**Cara pakai di notebook 05 (HPO).** Pilih salah satu: (a) latih di train, pilih hiperparameter di valid; atau (b) CV 5-fold di train+valid memakai `cv_fold`. Model terpilih dilatih ulang di train+valid, lalu **dinilai sekali** di test.

In [ ]:
sp_eda = pd.read_csv(INTERIM / 'split.csv').set_index('company_id').split
lab = rl.merge(co[['company_id', 'business_group']], on='company_id').set_index('company_id')
lab['grp'] = lab.business_group.fillna(pd.Series(lab.index, index=lab.index))
dev = lab[sp_eda.reindex(lab.index) == 'dev']
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_ix, va_ix = next(sgkf.split(dev, dev.risk_class, dev.grp))
split = pd.Series('unlabeled', index=co.company_id, name='split')
split[dev.index[tr_ix]] = 'train'; split[dev.index[va_ix]] = 'valid'
split[sp_eda[sp_eda == 'holdout'].index] = 'test'
cv_fold = pd.Series(-1, index=co.company_id, name='cv_fold')
for k, (_, ix) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(dev, dev.risk_class, dev.grp)):
    cv_fold[dev.index[ix]] = k
SPLIT = pd.concat([split, cv_fold], axis=1).reset_index()
SPLIT.to_csv(PROC / 'split.csv', index=False)
tab = pd.crosstab(split.reindex(lab.index), lab.risk_class)[RISK_ORDER]; tab['n'] = tab.sum(axis=1); tab['% High'] = (100 * tab.High / tab.n).round(1)
assert set(split[split == 'test'].index) == set(sp_eda[sp_eda == 'holdout'].index)
grp_split = lab.dropna(subset=['business_group']).assign(s=split).groupby('business_group').s.nunique().gt(1).sum()
print(f'test identik dengan holdout notebook 02 ✓ · grup usaha yang terbelah antar split: {grp_split}')
display(tab.reindex(['train', 'valid', 'test']))
display(pd.crosstab(SPLIT.cv_fold[SPLIT.cv_fold >= 0].values, lab.risk_class.reindex(SPLIT.company_id[SPLIT.cv_fold >= 0]).values, rownames=['cv_fold'], colnames=['risk_class'])[RISK_ORDER])
FIT_MASK = split.isin(['train', 'unlabeled'])       # baris yang boleh dipakai untuk fit statistik tanpa label

## C. Fitur risiko — satu baris per perusahaan (per 30 Jun 2026)
Fitur dibangun per kelompok. Setiap kelompok mencatat **sumber temuan** di notebook 02 dan **tingkat bukti** (A/B/C). Fitur transaksi (timing, arus kas) hanya diisi untuk nasabah. Untuk prospect nilainya dibiarkan **kosong (NaN)**, bukan 0, karena transaksinya hanya terlihat sebagian (G2). Nilai 0 akan terbaca model sebagai "tidak pernah telat" atau "tidak ada arus kas".

In [ ]:
DICT = []        # kamus fitur: diisi oleh setiap kelompok
def register(cols, kelompok, sumber, tingkat, definisi, portabel):
    for c in ([cols] if isinstance(cols, str) else cols):
        DICT.append(dict(fitur=c, kelompok=kelompok, sumber_EDA=sumber, tingkat_bukti=tingkat, definisi=definisi, set_portabel=portabel))

R = pd.DataFrame(index=pd.Index(co.company_id, name='company_id'))
prof = co.set_index('company_id')
R['industry'] = prof.industry; R['size_class_reference'] = prof.size_class_reference; R['customer_type'] = prof.customer_type
R = R.join(SPLIT.set_index('company_id'))
R['risk_class'] = rl.set_index('company_id').risk_class            # target (kosong untuk tanpa label)

# --- C1. Keuangan: bahan mentah + relatif-peer (fit di train + unlabeled) + tren + missing-sebagai-sinyal ---
fy = {y: fin[fin.fiscal_year == y].set_index('company_id') for y in (2024, 2025)}
slog = lambda v: np.sign(v) * np.log1p(np.abs(v))
for y, f_ in fy.items():
    sfx = '' if y == 2025 else '_2024'
    R['fin_log_der' + sfx] = np.log((f_.total_debt_idr_bn / f_.total_equity_idr_bn).clip(lower=1e-3))
    R['fin_log_cr' + sfx] = np.log((f_.current_assets_idr_bn / f_.current_liabilities_idr_bn).clip(lower=1e-3))
    R['fin_npm' + sfx] = f_.net_profit_idr_bn / f_.revenue_idr_bn
    R['fin_sicr' + sfx] = slog(f_.ebit_idr_bn / f_.interest_expense_idr_bn)
    R['fin_log_revenue' + sfx] = np.log(f_.revenue_idr_bn)
R['fin_log_assets'] = np.log(fy[2025].total_assets_idr_bn)
R['d_log_der'] = R.fin_log_der - R.fin_log_der_2024; R['d_npm'] = R.fin_npm - R.fin_npm_2024; R['rev_growth'] = R.fin_log_revenue - R.fin_log_revenue_2024
R = R.drop(columns=[c for c in R if c.endswith('_2024')])
R['has_report'] = R.fin_log_revenue.notna().astype(int)
R['is_audited'] = fy[2025].is_audited.reindex(R.index).astype(float).fillna(0).astype(int)
R['ebit_missing'] = (R.has_report.eq(1) & fy[2025].ebit_idr_bn.reindex(R.index).isna()).astype(int)
R['interest_missing'] = (R.has_report.eq(1) & fy[2025].interest_expense_idr_bn.reindex(R.index).isna()).astype(int)
pz = PeerZScaler().fit(R[FIT_MASK.reindex(R.index).values][RATIO_COLS + PEER_KEYS])
R = R.join(pz.transform(R[RATIO_COLS + PEER_KEYS]))
register(['der_pz', 'npm_pz', 'icr_pz'], 'keuangan relatif-peer', 'F1, D3', 'A', 'z robust per industri×ukuran (back-off ke industri/global bila < 10), fit train+unlabeled, clip ±6', True)
register('cr_pz', 'keuangan relatif-peer', 'F1', 'B', 'idem untuk log CR (F1: tidak lebih baik dari mentah → bandingkan)', True)
register(['fin_log_der', 'fin_log_cr', 'fin_npm', 'fin_sicr'], 'keuangan mentah', 'F1, D3', 'A', 'log DER, log CR, NPM, sign·log1p(ICR) FY2025', True)
register(['fin_log_revenue', 'fin_log_assets'], 'keuangan mentah', 'G2', 'C', 'ukuran usaha dari laporan', True)
register(['d_log_der', 'd_npm', 'rev_growth'], 'tren keuangan', '—', 'C', 'perubahan FY2025 − FY2024', True)
register(['has_report', 'is_audited'], 'missing-sebagai-sinyal', 'D2', 'A', 'laporan FY2025 tersedia / diaudit (MNAR: High 49% vs 18%)', True)
register(['ebit_missing', 'interest_missing'], 'missing-sebagai-sinyal', 'D2', 'C', 'item laporan kosong (MCAR → kemungkinan lemah)', True)
print('statistik peer di-fit pada', int(FIT_MASK.sum()), 'perusahaan (train + unlabeled); contoh median DER per industri×ukuran:')
display(pd.DataFrame({k: (np.exp(v[0]), v[2]) for k, v in list(pz.stats_['fin_log_der'][0].items())[:6]}, index=['DER median', 'n']).T.round(2))

In [ ]:
# --- C2. Timing pembayaran (F2) & C3. arus kas terkoreksi musim (F2, F5) — hanya nasabah ---
pay = tx[tx.transaction_type == 'Payroll']; tax = tx[tx.transaction_type == 'Tax']
def share_window(df, cond, m0, m1):
    d = df[(df.ym >= pd.Period(m0, 'M')) & (df.ym <= pd.Period(m1, 'M'))]
    return d.assign(c=cond(d)).groupby('company_id').c.mean()
late_pay = lambda d: d.day >= 28; late_tax = lambda d: d.day > 15
R['pay_late_6m'] = share_window(pay, late_pay, '2026-01', '2026-06'); R['pay_late_12m'] = share_window(pay, late_pay, '2025-07', '2026-06')
R['tax_late_6m'] = share_window(tax, late_tax, '2026-01', '2026-06'); R['tax_late_12m'] = share_window(tax, late_tax, '2025-07', '2026-06')
R['pay_late_30m'] = share_window(pay, late_pay, '2024-01', '2026-06'); R['tax_late_30m'] = share_window(tax, late_tax, '2024-01', '2026-06')
R['pay_split_12m'] = pay[pay.ym >= pd.Period('2025-07', 'M')].groupby(['company_id', 'ym']).size().gt(1).groupby('company_id').mean()
lt = pd.concat([pay.assign(l=late_pay(pay))[['company_id', 'ym', 'l']], tax.assign(l=late_tax(tax))[['company_id', 'ym', 'l']]])
LATE = lt.groupby(['company_id', 'ym']).l.mean().unstack().reindex(columns=MONTHS)
R['timing_delta'] = LATE.loc[:, '2026-01':'2026-06'].mean(axis=1) - LATE.loc[:, '2025-01':'2025-12'].mean(axis=1)
register(['pay_late_6m', 'pay_late_12m', 'tax_late_6m', 'tax_late_12m'], 'timing bayar', 'F2', 'A', '% gaji dibayar ≥ tgl 28 / pajak disetor > tgl 15, jendela 6 & 12 bln', False)
register(['pay_late_30m', 'tax_late_30m'], 'timing bayar', 'F2', 'A', 'idem, seluruh riwayat 30 bln (kebiasaan kronis; AUC EDA 0,85–0,86)', False)
register('pay_split_12m', 'timing bayar', 'F2', 'A', '% bulan gaji dicicil (> 1 batch), 12 bln', False)
register('timing_delta', 'timing bayar', 'F2, F10h', 'B', '% telat (gaji+pajak) 6 bln terakhir − 12 bln sebelumnya', False)

INN = tx[tx.direction == 'IN'].groupby(['company_id', 'ym']).amount_idr_mn.sum().unstack(fill_value=0).reindex(columns=MONTHS, fill_value=0)
OUTF = tx[tx.direction == 'OUT'].groupby(['company_id', 'ym']).amount_idr_mn.sum().unstack(fill_value=0).reindex(columns=MONTHS, fill_value=0)
# faktor musiman industri × bulan kalender-relatif (F5): median residual log inflow, fit pada nasabah train + unlabeled
Lc = np.log1p(INN.reindex(CLIENTS)); Ldm = Lc.sub(Lc.median(axis=1), axis=0)
ind_c = prof.industry.reindex(Ldm.index); fitc = FIT_MASK.reindex(Ldm.index).values
SEAS = Ldm[fitc].groupby(ind_c[fitc]).median()                         # industri × bulan (menangkap Lebaran bergerak per tahun)
L_sa = Lc - SEAS.reindex(ind_c.values).set_axis(Lc.index)               # log inflow terkoreksi musim
R['inflow_cv_sa'] = np.exp(L_sa.loc[:, '2025-07':'2026-06']).pipe(lambda x: x.std(axis=1) / x.mean(axis=1))
R['inflow_growth_sa'] = L_sa.loc[:, '2026-01':'2026-06'].mean(axis=1) - L_sa.loc[:, '2025-01':'2025-06'].mean(axis=1)
zc = L_sa.sub(L_sa.median(axis=1), axis=0); zc = zc.div(1.4826 * zc.abs().median(axis=1).replace(0, np.nan), axis=0)
R['max_inflow_z_12m'] = zc.loc[:, '2025-07':'2026-06'].max(axis=1)
in6, in12 = INN.loc[:, '2026-01':'2026-06'].sum(axis=1), INN.loc[:, '2025-07':'2026-06'].sum(axis=1)
R['out_in_6m'] = (OUTF.loc[:, '2026-01':'2026-06'].sum(axis=1) / in6).reindex(CLIENTS)
R['out_in_12m'] = (OUTF.loc[:, '2025-07':'2026-06'].sum(axis=1) / in12).reindex(CLIENTS)
rev25 = np.exp(R.fin_log_revenue)
R['capture_ratio'] = (INN.loc[:, '2025-01':'2025-12'].sum(axis=1).reindex(R.index) / (rev25 * 1000)).where(IS_CLIENT)
R['divergence'] = (np.log(in6 / INN.loc[:, '2025-01':'2025-06'].sum(axis=1)).reindex(R.index) - R.rev_growth).where(IS_CLIENT)
for c in ['pay_late_6m', 'pay_late_12m', 'tax_late_6m', 'tax_late_12m', 'pay_late_30m', 'tax_late_30m', 'pay_split_12m', 'timing_delta', 'inflow_cv_sa', 'inflow_growth_sa', 'max_inflow_z_12m', 'out_in_6m', 'out_in_12m']:
    R[c] = R[c].where(IS_CLIENT)                                          # prospect: NaN (visibilitas parsial)
register(['out_in_6m', 'out_in_12m'], 'arus kas', 'F2, G1', 'A', 'outflow ÷ inflow 6 & 12 bln', False)
register(['inflow_cv_sa', 'inflow_growth_sa'], 'arus kas', 'F5', 'B', 'CV & pertumbuhan inflow SETELAH koreksi musim industri×bulan (fit train+unlabeled)', False)
register('max_inflow_z_12m', 'arus kas', 'D3', 'B', 'z robust lonjakan inflow bulanan maksimum 12 bln (setelah koreksi musim)', False)
register(['capture_ratio', 'divergence'], 'arus kas', 'F6', 'A', 'inflow BNI ÷ omzet 2025; Δlog inflow H1 − Δlog omzet', False)
print('faktor musiman (industri × bulan) — puncak per industri:')
display(SEAS.idxmax(axis=1).astype(str).rename('bulan puncak').to_frame().join(SEAS.max(axis=1).round(2).rename('efek log')))

**Catatan detektor anchor online.** Di notebook 02 (retrospektif, 18 bulan data sebelum guncangan), patahan anchor terlihat sangat jelas (z −9 dan −5). Versi online hanya boleh memakai data sampai bulan berjalan, sehingga sinyalnya lebih lemah: z terendah C0409 hanya ±−2,9. Karena itu ambang online diturunkan ke **−2,5**. Tabel di bawah sel ini menunjukkan semua anchor lain tidak pernah turun di bawah −2,0, sehingga hasilnya sama untuk ambang −2,0 sampai −2,5, dan ambang ini tidak menghasilkan alarm palsu di data ini. Ambang ini dicatat sebagai parameter yang perlu dikalibrasi ulang bila data bertambah.

In [ ]:
# --- C4. Fasilitas (G1) · C5. Konsentrasi & relasi (F3) · C6. Paparan anchor (F4) ---
cfi = cf.set_index('company_id')
R['utilization'] = cfi.utilization; R['log_limit'] = np.log(cfi.limit_idr_bn)
for v in cfi.facility_type.dropna().unique(): R['fac_' + re.sub(r'\W+', '_', v.lower()).strip('_')] = (cfi.facility_type == v).reindex(R.index).astype(float)
R['policy_restricted'] = cfi.policy_restriction.notna().reindex(R.index).astype(float)
fac_cols = [c for c in R if c.startswith('fac_')]
register(['utilization', 'log_limit'], 'fasilitas', 'G1', 'A', 'outstanding ÷ limit; log limit (BUKAN 4 kolom proxy-label)', False)
register(fac_cols + ['policy_restricted'], 'fasilitas', 'D4', 'C', 'jenis fasilitas (one-hot), ada pembatasan kebijakan', False)

act = rel[(rel.relationship_start <= CUTOFF) & (rel.relationship_end.isna() | (rel.relationship_end > CUTOFF))]
buy = act[act.relationship_type.isin(['Customer', 'Distributor'])]; sup = act[act.relationship_type == 'Supplier']
R['top_buyer_share'] = buy.groupby('source_id').dependency_share.max()
R['buyer_hhi'] = buy.assign(s2=buy.dependency_share ** 2).groupby('source_id').s2.sum()
R['n_buyers'] = buy.groupby('source_id').size(); R['n_suppliers'] = sup.groupby('source_id').size()
R['top_supplier_share'] = sup.groupby('source_id').dependency_share.max()
for c in ['top_buyer_share', 'buyer_hhi', 'n_buyers', 'n_suppliers', 'top_supplier_share']: R[c] = R[c].fillna(0)
register('top_buyer_share', 'konsentrasi', 'F3', 'A', 'porsi pembeli terbesar dari relasi aktif (kontinu, TANPA ambang)', True)
register(['buyer_hhi', 'n_buyers', 'n_suppliers', 'top_supplier_share'], 'konsentrasi', 'F3', 'C', 'HHI pembeli, jumlah pembeli/pemasok, porsi pemasok terbesar', True)

anchors = co.loc[co.is_anchor_candidate, 'company_id'].tolist(); xm = np.arange(T)
ANCHOR_Z_THR = -2.5
def anchor_z(a, t_end, minpre=12):
    """Detektor ONLINE patahan tren anchor (F4): tren Theil–Sen pada inflow terkoreksi musim dari awal data s.d. t−3,
    lalu z = median residual 3 bulan terakhir (t−2…t) ÷ MAD residual. Hanya memakai data ≤ t."""
    yv = (np.log1p(INN.loc[a]) - SEAS.loc[prof.industry[a]].values).values; hi_ = t_end - 3
    if hi_ + 1 < minpre: return np.nan
    sl, ic, *_ = stats.theilslopes(yv[:hi_ + 1], xm[:hi_ + 1]); res = yv - (ic + sl * xm)
    mad_ = 1.4826 * np.median(np.abs(res[:hi_ + 1] - np.median(res[:hi_ + 1])))
    return np.median(res[t_end - 2:t_end + 1]) / mad_ if mad_ > 0 else np.nan
AZ = pd.DataFrame({a: [anchor_z(a, t) for t in range(T)] for a in anchors}, index=MONTHS).T     # anchor × bulan
SHOCK = AZ.lt(ANCHOR_Z_THR)
SHOCKED = SHOCK.loc[:, '2025-07':].any(axis=1).pipe(lambda s_: s_[s_].index.tolist())
r_jul = rel[(rel.relationship_start <= '2025-07-01') & rel.relationship_type.isin(['Customer', 'Distributor'])]
R['exp_shocked_anchor'] = r_jul[r_jul.target_id.isin(SHOCKED)].groupby('source_id').dependency_share.sum()
R['exp_shocked_anchor'] = R.exp_shocked_anchor.fillna(0).where(~R.index.isin(anchors))
register('exp_shocked_anchor', 'jaringan', 'F4', 'A', 'Σ dependency_share ke anchor yang patah tren (detektor online, z < −2,5) sejak Jul 2025', True)
print('anchor yang terdeteksi patah tren (online, sampai cutoff):', SHOCKED, '| sensitivitas ambang:',
      {thr: int(AZ.loc[:, '2025-07':].lt(thr).any(axis=1).sum()) for thr in (-2.0, -2.5, -3.0, -3.5)})
display(AZ.loc[:, '2025-06':].round(1))

In [ ]:
# --- C7. Profil & geografi (D4, F7) · C8. Sentralitas jaringan (F8) ---
R['size_ord'] = R.size_class_reference.map({'Kecil': 0, 'Menengah': 1, 'Besar': 2})
R['age'] = 2026 - prof.establishment_year; R['log_emp'] = np.log(prof.employees)
R['is_listed'] = prof.is_listed.astype(int); R['has_import'] = prof.has_import_activity.astype(int); R['payroll_bni'] = (prof.payroll_bank == 'BNI').astype(int)
gsize = prof.business_group.value_counts(); R['group_size'] = prof.business_group.map(gsize).fillna(0); R['in_group'] = (R.group_size > 0).astype(int)
for v in sorted(prof.industry.unique()): R['ind_' + re.sub(r'\W+', '_', v.lower()).strip('_')] = (prof.industry == v).astype(int)
for v in sorted(prof.city.unique()): R['city_' + re.sub(r'\W+', '_', v.lower()).strip('_')] = (prof.city == v).astype(int)
ind_cols = [c for c in R if c.startswith('ind_')]; city_cols = [c for c in R if c.startswith('city_')]
register(ind_cols + ['size_ord'], 'profil', 'E2, D4', 'A', 'industri one-hot; ukuran ordinal Kecil<Menengah<Besar', True)
register(city_cols, 'profil', 'F8, D4', 'C', 'kota one-hot (komunitas jaringan mengikuti kota)', True)
register(['age', 'log_emp', 'is_listed', 'has_import', 'payroll_bni', 'group_size', 'in_group'], 'profil', 'G1, D4', 'C', 'umur, log karyawan, tercatat bursa, impor, payroll di BNI, grup usaha', True)

def hav(a1, o1, a2, o2):
    a1, o1, a2, o2 = map(np.radians, [a1, o1, a2, o2]); return 6371 * 2 * np.arcsin(np.sqrt(np.sin((a2 - a1) / 2) ** 2 + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2))
R['dist_branch_km'] = hav(prof.latitude.values[:, None], prof.longitude.values[:, None], br.latitude.values[None], br.longitude.values[None]).min(axis=1)
loc = prof.location_id; cl_ = IS_CLIENT.astype(int)
n_loc, k_loc = loc.map(loc.value_counts()), loc.map(cl_.groupby(loc).sum())
R['district_penetration'] = (k_loc - cl_) / (n_loc - 1)                 # leave-one-out: tanpa dirinya sendiri
g_ = geo.set_index('location_id'); R['district_poi'] = loc.map(g_.poi_industrial_count); R['district_has_branch'] = loc.isin(br.location_id).astype(int)
register(['district_penetration', 'district_poi', 'district_has_branch', 'dist_branch_km'], 'geografi', 'F7, D4', 'C', 'atribut kecamatan pengganti one-hot 40 level (penetrasi leave-one-out)', True)

G = nx.Graph(); G.add_edges_from(act[['source_id', 'target_id']].values)
R['degree'] = pd.Series(dict(G.degree())); R['pagerank'] = pd.Series(nx.pagerank(G)) * len(G); R['core_number'] = pd.Series(nx.core_number(G))
ne_non = set(ne.loc[~ne.is_bni_customer, 'entity_id']) | set(co.loc[co.customer_type == 'Prospect', 'company_id'])
R['n_nonbni_partners'] = act[act.target_id.isin(ne_non)].groupby('source_id').target_id.nunique()
for c in ['degree', 'pagerank', 'core_number', 'n_nonbni_partners']: R[c] = R[c].fillna(0)
register(['degree', 'pagerank', 'core_number', 'n_nonbni_partners'], 'jaringan', 'F8', 'C', 'sentralitas graf relasi aktif; jumlah mitra non-BNI', True)

DICT_DF = pd.DataFrame(DICT)
missing_cols = [c for c in DICT_DF.fitur if c not in R]; assert not missing_cols, missing_cols
print(f'features_risk: {R.shape[0]:,} baris × {len(DICT_DF)} fitur ({DICT_DF.set_portabel.sum()} portabel) + kolom kunci')
display(DICT_DF.groupby(['kelompok', 'tingkat_bukti']).size().unstack(fill_value=0))

## D. Fitur panel anomali — satu baris per nasabah × bulan (online)
Panel berisi 1.400 nasabah (non-prospect) × 30 bulan. Prospect tidak dimasukkan karena transaksinya hanya terlihat sebagian. Setiap fitur di bulan *t* **hanya memakai data sampai *t*** supaya bisa dipakai untuk alarm bulanan yang sebenarnya:

| Fitur | Jenis kejadian (peta F10g) | Cara online |
|---|---|---|
| `inflow_resid_z` | dasar semua jenis | residual = log inflow − faktor industri bulan *t* (median penampang lintang nasabah lain di bulan yang sama); z memakai median/MAD residual bulan < *t* (min. 6 bulan) |
| `spike_candidate`, `spike_reverted` | lonjakan sesaat | z ≥ 3,5 setelah 2 bulan tenang; konfirmasi bulan berikutnya bila z turun lagi |
| `chg_3v6` | erosi, distress, ekspansi | rata-rata residual 3 bln terakhir − 6 bln sebelumnya |
| `cusum_down`, `cusum_up` | perubahan level bertahan | CUSUM Page (1954) pada z, *k* = 0,5 |
| `buyer_exit_share`, `n_active_cp_3m`, `cp_change` | pembeli utama keluar | counterparty aktif 12 bln sebelumnya lalu diam 2 bln; **tanpa** `relationship_end` |
| `exp_shocked_anchor_t` | efek berantai | paparan relasi (yang sudah dimulai) ke anchor yang terdeteksi patah di bulan *t* |
| `late_share_3m`, `timing_delta_t`, `timing_jump_3v9` | distress (lead ±3 bln) | % gaji/pajak telat 3 bln; 6 bln − 12 bln sebelumnya (lebih tepat); 3 bln − 9 bln sebelumnya (lebih cepat) |
| `out_in_3m`, `out_in_delta` | distress | rasio outflow/inflow 3 bln vs 12 bln sebelumnya |
| `capture_t`, `capture_chg_6m` | erosi wallet | inflow 12 bln ÷ omzet tahun buku terakhir **yang sudah terbit** (lag 4 bln) |

In [ ]:
PNL_IDS = CLIENTS; nC = len(PNL_IDS)
Lp = np.log1p(INN.reindex(PNL_IDS).values)                                       # nasabah × bulan
indP = prof.industry.reindex(PNL_IDS).values
def expanding_stat(A, t, fn):
    return fn(A[:, :t], axis=1) if t > 0 else np.full(A.shape[0], np.nan)
# 1) faktor industri × bulan online: median (log inflow_t − median log inflow sebelumnya) antar nasabah se-industri di bulan t
dev_t = np.full_like(Lp, np.nan)
for t in range(6, T): dev_t[:, t] = Lp[:, t] - np.nanmedian(Lp[:, :t], axis=1)
fac = pd.DataFrame(dev_t).groupby(indP).transform('median').values
Rp = Lp - np.nan_to_num(fac)                                                     # residual terkoreksi industri × bulan
Zp = np.full_like(Rp, np.nan)
for t in range(6, T):
    med = np.nanmedian(Rp[:, :t], axis=1); mad = 1.4826 * np.nanmedian(np.abs(Rp[:, :t] - med[:, None]), axis=1)
    Zp[:, t] = (Rp[:, t] - med) / np.where(mad > 0, mad, np.nan)
Zc = np.clip(Zp, -10, 10)
# 2) spike, perubahan 3v6, CUSUM
prev2 = np.full_like(Zc, np.nan); prev2[:, 2:] = np.fmax(Zc[:, :-2], Zc[:, 1:-1])
spike_cand = (Zc >= 3.5) & (prev2 < 2)
spike_rev = np.zeros_like(spike_cand); spike_rev[:, 1:] = (Zc[:, :-1] >= 3.5) & (Zc[:, 1:] < 1.5)
Rdf = pd.DataFrame(Rp)
chg_3v6 = (Rdf.T.rolling(3).mean() - Rdf.T.shift(3).rolling(6).mean()).T.values
cus_dn = np.zeros_like(Zc); cus_up = np.zeros_like(Zc); K = .5
for t in range(1, T):
    z_ = np.nan_to_num(Zc[:, t]); cus_dn[:, t] = np.maximum(0, cus_dn[:, t - 1] - z_ - K); cus_up[:, t] = np.maximum(0, cus_up[:, t - 1] + z_ - K)
# 3) pembeli berhenti (tanpa relationship_end)
ibk = tx[(tx.direction == 'IN') & tx.company_id.isin(PNL_IDS) & tx.counterparty_id.notna() & tx.counterparty_id.astype(str).str.match(r'^[CE]\d')]
cpm = ibk.groupby(['company_id', 'counterparty_id', 'ym']).amount_idr_mn.sum().unstack(fill_value=0).reindex(columns=MONTHS, fill_value=0)
A_ = cpm.values; pos_ = pd.Index(PNL_IDS).get_indexer(cpm.index.get_level_values(0))
TOT = INN.reindex(PNL_IDS).values
csA = np.concatenate([np.zeros((len(A_), 1)), A_.cumsum(axis=1)], axis=1); csT = np.concatenate([np.zeros((nC, 1)), TOT.cumsum(axis=1)], axis=1)
bx = np.zeros((nC, T)); nact = np.zeros((nC, T)); nact_prev = np.zeros((nC, T))
for t in range(T):
    lo_ = max(0, t - 13); hi_ = t - 1                                           # jendela aktif: t−13 … t−2
    if hi_ - lo_ < 3: continue
    act_win = csA[:, hi_] - csA[:, lo_]; silent = (A_[:, t] == 0) & (A_[:, t - 1] == 0)
    share = act_win / np.maximum(csT[pos_, hi_] - csT[pos_, lo_], 1e-9)
    val = np.where(silent & (act_win > 0), share, 0.0)
    np.maximum.at(bx[:, t], pos_, val)
    np.add.at(nact[:, t], pos_, (A_[:, max(0, t - 2):t + 1].sum(axis=1) > 0).astype(float))
    if t >= 5: np.add.at(nact_prev[:, t], pos_, (A_[:, t - 5:t - 2].sum(axis=1) > 0).astype(float))
# 4) paparan anchor bermasalah online (relasi yang SUDAH dimulai; tanpa relationship_end)
rb = rel[rel.relationship_type.isin(['Customer', 'Distributor']) & rel.target_id.isin(anchors) & rel.source_id.isin(PNL_IDS)]
exp_t = np.zeros((nC, T)); ip = pd.Index(PNL_IDS)
for t, m_ in enumerate(MONTHS):
    bad = SHOCK.index[SHOCK.loc[:, :m_].any(axis=1)] if t >= 12 else []                # "pernah patah s.d. bulan t"
    r_ = rb[(rb.relationship_start <= m_.end_time) & rb.target_id.isin(bad)]
    if len(r_): s_ = r_.groupby('source_id').dependency_share.sum(); exp_t[ip.get_indexer(s_.index), t] = s_.values
# 5) timing, rasio arus kas, capture dengan lag laporan
LT = LATE.reindex(PNL_IDS).values
late3 = pd.DataFrame(LT).T.rolling(3, min_periods=2).mean().T.values
td = (pd.DataFrame(LT).T.rolling(6, min_periods=4).mean() - pd.DataFrame(LT).T.shift(6).rolling(12, min_periods=8).mean()).T.values
tj = (pd.DataFrame(LT).T.rolling(3, min_periods=2).mean() - pd.DataFrame(LT).T.shift(3).rolling(9, min_periods=6).mean()).T.values   # versi cepat (lead lebih awal, alarm palsu lebih banyak)
OUTP = OUTF.reindex(PNL_IDS).fillna(0).values
r3 = lambda A: pd.DataFrame(A).T.rolling(3).sum().T.values
oi3 = r3(OUTP) / np.where(r3(TOT) > 0, r3(TOT), np.nan)
oi12p = (pd.DataFrame(OUTP).T.shift(3).rolling(12).sum() / pd.DataFrame(TOT).T.shift(3).rolling(12).sum().replace(0, np.nan)).T.values
rev_pivot = fin.pivot(index='company_id', columns='fiscal_year', values='revenue_idr_bn').reindex(PNL_IDS)
in12 = pd.DataFrame(TOT).T.rolling(12).sum().T.values
cap = np.full((nC, T), np.nan)
for t, m_ in enumerate(MONTHS):
    fy_av = [y for y in rev_pivot.columns if pd.Period(f'{y + 1}-{REPORT_LAG_MONTHS:02d}', 'M') <= m_]
    if fy_av: cap[:, t] = in12[:, t] / (rev_pivot[max(fy_av)].values * 1000)
cap_chg = cap - np.concatenate([np.full((nC, 6), np.nan), cap[:, :-6]], axis=1)

cp_chg = nact - nact_prev; cp_chg[:, :6] = np.nan; bx[:, :4] = np.nan
PNL = pd.DataFrame({'company_id': np.repeat(PNL_IDS, T), 'ym': np.tile(MONTHS.astype(str), nC)})
for nm, A in [('inflow_log', Lp), ('inflow_resid', Rp), ('inflow_resid_z', Zp), ('spike_candidate', spike_cand.astype(int)), ('spike_reverted', spike_rev.astype(int)),
              ('chg_3v6', chg_3v6), ('cusum_down', cus_dn), ('cusum_up', cus_up), ('buyer_exit_share', bx), ('n_active_cp_3m', nact), ('cp_change', cp_chg),
              ('exp_shocked_anchor_t', exp_t), ('late_share_3m', late3), ('timing_delta_t', td), ('timing_jump_3v9', tj), ('out_in_3m', oi3), ('out_in_delta', oi3 - oi12p),
              ('capture_t', cap), ('capture_chg_6m', cap_chg)]:
    PNL[nm] = np.asarray(A, dtype=float).ravel()
PNL = PNL.merge(SPLIT, on='company_id', how='left').merge(co[['company_id', 'industry']], on='company_id')
ANOM_COLS = [c for c in PNL.columns if c not in ('company_id', 'ym', 'split', 'cv_fold', 'industry', 'inflow_log', 'inflow_resid')]
print(f'features_anomaly_panel: {len(PNL):,} baris ({nC} nasabah × {T} bulan) × {len(ANOM_COLS)} fitur')
print('6 bulan pertama = masa "pemanasan" (statistik online belum cukup) → z bernilai NaN; model sebaiknya mulai dari 2024-07.')
display(PNL[ANOM_COLS].describe().T[['count', 'mean', '50%', 'max']].round(3))
for c, k, tg, d in [('inflow_resid_z', 'panel', 'F10a', 'B'), ('spike_candidate', 'panel', 'F10c', 'B'), ('spike_reverted', 'panel', 'F10c', 'B'), ('chg_3v6', 'panel', 'F10g', 'A'),
                    ('cusum_down', 'panel', 'F10c', 'B'), ('cusum_up', 'panel', 'F10c', 'B'), ('buyer_exit_share', 'panel', 'F10e', 'A'), ('n_active_cp_3m', 'panel', 'F10e', 'C'),
                    ('cp_change', 'panel', 'F10e', 'C'), ('exp_shocked_anchor_t', 'panel', 'F4', 'A'), ('late_share_3m', 'panel', 'F2', 'A'), ('timing_delta_t', 'panel', 'F10h', 'A'), ('timing_jump_3v9', 'panel', 'F10h', 'B'),
                    ('out_in_3m', 'panel', 'F2', 'B'), ('out_in_delta', 'panel', 'F10g', 'B'), ('capture_t', 'panel', 'F6', 'B'), ('capture_chg_6m', 'panel', 'F6', 'B')]:
    DICT.append(dict(fitur=c, kelompok='anomali (panel)', sumber_EDA=tg, tingkat_bukti=d, definisi='lihat tabel bagian D', set_portabel=False))

## E. Uji nilai fitur
### E1 · Ablation kumulatif untuk klasifikasi risiko (CV berulang di train+valid)
Kelompok fitur ditambahkan satu per satu, dan setiap langkah diukur dengan **macro-F1** dan **recall kelas High**. Validasinya *repeated stratified group CV*: 2 pengulangan × 5 fold (pengulangan pertama = `cv_fold` resmi) di train+valid (800 nasabah). Test tidak disentuh.

Dua model dipakai supaya kesimpulannya tidak bergantung pada satu algoritma:
- logistic regression (impute + standardisasi);
- gradient boosting histogram.

Keduanya memakai `class_weight='balanced'` karena High hanya ±14%. Fitur relatif-peer dihitung dengan `PeerZScaler` **di dalam pipeline**, jadi statistik peer di-fit ulang di setiap fold tanpa melihat fold validasinya.

In [ ]:
GROUPS = {
    'profil':                ind_cols + city_cols + ['size_ord', 'age', 'log_emp', 'is_listed', 'has_import', 'payroll_bni', 'in_group', 'group_size'],
    'keuangan mentah':       ['fin_log_der', 'fin_log_cr', 'fin_npm', 'fin_sicr', 'fin_log_revenue', 'fin_log_assets'],
    'keuangan relatif-peer': ['der_pz', 'cr_pz', 'npm_pz', 'icr_pz'],
    'missing & tren':        ['has_report', 'is_audited', 'ebit_missing', 'interest_missing', 'd_log_der', 'd_npm', 'rev_growth'],
    'timing bayar':          ['pay_late_6m', 'pay_late_12m', 'pay_late_30m', 'tax_late_6m', 'tax_late_12m', 'tax_late_30m', 'pay_split_12m', 'timing_delta'],
    'arus kas':              ['out_in_6m', 'out_in_12m', 'inflow_cv_sa', 'inflow_growth_sa', 'max_inflow_z_12m', 'capture_ratio', 'divergence'],
    'fasilitas':             ['utilization', 'log_limit'] + fac_cols + ['policy_restricted'],
    'konsentrasi':           ['top_buyer_share', 'buyer_hhi', 'n_buyers', 'n_suppliers', 'top_supplier_share'],
    'jaringan & geografi':   ['exp_shocked_anchor', 'degree', 'pagerank', 'core_number', 'n_nonbni_partners', 'district_penetration', 'district_poi', 'district_has_branch', 'dist_branch_km'],
}
PZ_OUT = list(PeerZScaler().get_feature_names_out())
DEVSET = R[R.split.isin(['train', 'valid'])].copy(); y_dev = DEVSET.risk_class.map({'Low': 0, 'Medium': 1, 'High': 2}).values
grp_dev = co.set_index('company_id').business_group.reindex(DEVSET.index).fillna(pd.Series(DEVSET.index, index=DEVSET.index)).values

def make_model(cols, kind):
    """Pipeline: PeerZScaler (bila perlu) + kolom lain → LR atau HGB. Kolom pz dihitung ulang di dalam setiap fold."""
    use_pz = any(c in PZ_OUT for c in cols); other = [c for c in cols if c not in PZ_OUT]
    parts = ([('pz', PeerZScaler(), RATIO_COLS + PEER_KEYS)] if use_pz else []) + ([('rest', 'passthrough', other)] if other else [])
    ct = ColumnTransformer(parts, remainder='drop')
    if kind == 'LR':
        est = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(), LogisticRegression(max_iter=4000, C=.5, class_weight='balanced'))
    else:
        est = HistGradientBoostingClassifier(max_depth=3, max_leaf_nodes=15, learning_rate=.08, max_iter=150, max_bins=63, class_weight='balanced', random_state=SEED)   # versi ringan; HPO di notebook 05
    return Pipeline([('ct', ct), ('est', est)])

FOLDS = [list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(DEVSET, y_dev, grp_dev)) for r in range(2)]
FOLDS[0] = [(np.flatnonzero(DEVSET.cv_fold.values != k), np.flatnonzero(DEVSET.cv_fold.values == k)) for k in range(5)]   # pengulangan 1 = cv_fold resmi
def cv_score(cols, kind):
    f1s, recs = [], []
    with threadpool_limits(1):                       # data kecil: 1 thread lebih cepat dari banyak thread yang berebut
      for rep in FOLDS:
        for tr, va in rep:
            m = make_model(cols, kind).fit(DEVSET.iloc[tr], y_dev[tr]); p_ = m.predict(DEVSET.iloc[va])
            f1s.append(f1_score(y_dev[va], p_, average='macro')); recs.append(recall_score(y_dev[va], p_, labels=[2], average='macro'))
    return np.mean(f1s), np.std(f1s), np.mean(recs), np.std(recs)

t0 = time.time(); rows = []; cum = []
for g_, cols in GROUPS.items():
    cum = cum + cols
    for kind in ['LR', 'HGB']:
        f, fs, r_, rs = cv_score(cum, kind); rows.append(dict(langkah=f'+ {g_}', model=kind, n_fitur=len(cum), macro_F1=f, sd_F1=fs, recall_High=r_, sd_recall=rs))
ABL = pd.DataFrame(rows)
print(f'ablation kumulatif selesai dalam {time.time() - t0:.0f} dtk')
display(ABL.pivot(index='langkah', columns='model', values=['macro_F1', 'recall_High']).reindex([f'+ {g}' for g in GROUPS]).round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
steps = [f'+ {g}' for g in GROUPS]; yy = np.arange(len(steps))[::-1]
for a_, met, sd, ttl in [(ax[0], 'macro_F1', 'sd_F1', 'E1a · macro-F1 (CV 2×5, train+valid)'), (ax[1], 'recall_High', 'sd_recall', 'E1b · recall kelas High')]:
    for kind, col, off in [('LR', C['s1'], .17), ('HGB', C['s2'], -.17)]:
        d_ = ABL[ABL.model == kind].set_index('langkah').reindex(steps)
        a_.errorbar(d_[met], yy + off, xerr=d_[sd], fmt='o', color=col, ms=6, capsize=2, label='logistic regression' if kind == 'LR' else 'gradient boosting')
    a_.set_title(ttl); a_.set_xlim(.25, 1.0); a_.grid(axis='y', visible=False)
ax[0].axvline(.83, color=C['muted'], ls='--', lw=1); ax[0].text(.83, yy.max() + .45, ' oracle ≈ 0,83', fontsize=7.5, color=C['muted'])
ax[0].axvline(.718, color=C['s8'], ls=':', lw=1); ax[0].text(.718, -.75, 'probe EDA 0,72 ', fontsize=7.5, color=C['s8'], ha='right')
ax[0].set_yticks(yy); ax[0].set_yticklabels(steps); ax[1].legend(loc='upper center', bbox_to_anchor=(.5, -.2), ncol=2)
for a_ in ax: a_.set_xlabel('rata-rata ± simpangan baku 10 fold (angka persis di tabel atas)')
plt.tight_layout(); save_fig(fig, 'E1_ablation_cumulative'); plt.show()

### E2 · *Leave-one-group-out*: kelompok mana yang benar-benar diperlukan?
Ablation kumulatif bergantung pada urutan penambahan. Pemeriksaan pelengkapnya: mulai dari **set lengkap**, buang satu kelompok, lalu lihat seberapa besar skor turun. Kelompok yang membawa informasi unik akan menurunkan skor saat dibuang. Kelompok yang redundan (misalnya karena sama-sama mengukur faktor "kesehatan", G1 notebook 02) hampir tidak berpengaruh. Di sini juga dibandingkan rasio **mentah vs relatif-peer** dan **set portabel** untuk prospect.

In [ ]:
FULL = sum(GROUPS.values(), [])
PORTABLE = [c for c in FULL if c in set(DICT_DF.loc[DICT_DF.set_portabel, 'fitur'])]
base = {k: cv_score(FULL, k) for k in ['LR', 'HGB']}
rows = []
for g_, cols in list(GROUPS.items()) + [('rasio mentah (pertahankan peer)', GROUPS['keuangan mentah'][:4]), ('relatif-peer (pertahankan mentah)', GROUPS['keuangan relatif-peer'])]:
    keep = [c for c in FULL if c not in cols]
    for kind in ['LR', 'HGB']:
        f, fs, r_, rs = cv_score(keep, kind); rows.append(dict(dibuang=g_, model=kind, d_macro_F1=f - base[kind][0], d_recall_High=r_ - base[kind][2]))
LOGO = pd.DataFrame(rows)
port = {k: cv_score(PORTABLE, k) for k in ['LR', 'HGB']}
fig, ax = plt.subplots(figsize=(10, 4.6))
order = LOGO[LOGO.model == 'HGB'].sort_values('d_macro_F1').dibuang.tolist(); yy = np.arange(len(order))
for kind, col, off in [('LR', C['s1'], .18), ('HGB', C['s2'], -.18)]:
    d_ = LOGO[LOGO.model == kind].set_index('dibuang').reindex(order)
    ax.barh(yy + off, d_.d_macro_F1, height=.34, color=col, label='logistic regression' if kind == 'LR' else 'gradient boosting')
ax.axvline(0, color=C['ink2'], lw=.8); ax.set_yticks(yy); ax.set_yticklabels(order); ax.invert_yaxis(); ax.grid(axis='y', visible=False)
ax.set(xlabel='Δ macro-F1 bila kelompok DIBUANG dari set lengkap (negatif = kelompok itu penting)', title='E2 · Leave-one-group-out (CV 2×5)'); ax.legend(loc='upper center', bbox_to_anchor=(.5, -.14), ncol=2)
plt.tight_layout(); save_fig(fig, 'E2_leave_one_group_out'); plt.show()
summ = pd.DataFrame({k: dict(set_lengkap_F1=base[k][0], set_lengkap_recallHigh=base[k][2], set_portabel_F1=port[k][0], set_portabel_recallHigh=port[k][2]) for k in base}).T
display(summ.round(3)); display(LOGO.pivot(index='dibuang', columns='model', values=['d_macro_F1', 'd_recall_High']).round(3))
print(f'Set portabel ({len(PORTABLE)} fitur, tanpa transaksi & fasilitas) kehilangan {base["HGB"][0] - port["HGB"][0]:.3f} macro-F1 (HGB) dibanding set lengkap ({len(FULL)} fitur).')
print('Catatan: skor portabel di sini masih diukur pada NASABAH; kinerja pada prospect sebenarnya diukur di notebook 06 (gt_prospect_risk + importance weighting).')

### E2b · Pilih set awal berdasarkan **tingkat bukti**
Ablation memperlihatkan bahwa menambah semua fitur tidak otomatis lebih baik. Kelompok tingkat C (jaringan, geografi, kota) justru menambah derau. Karena itu dibandingkan tiga set: **A** (sudah diuji di EDA), **A+B** (ditambah turunan yang diuji di sini), dan **semua**. Simpangan baku antar-fold (±0,03) ikut ditampilkan, supaya selisih kecil tidak dibaca berlebihan. Dicek juga efek jendela timing: 6/12 bulan vs riwayat 30 bulan.

In [ ]:
DD = DICT_DF.set_index('fitur')
SET_A = [c for c in FULL if DD.tingkat_bukti.get(c) == 'A']; SET_AB = [c for c in FULL if DD.tingkat_bukti.get(c) in ('A', 'B')]
no30 = [c for c in SET_AB if not c.endswith('_30m')]
rows = []
for nm, cols in [('A', SET_A), ('A+B tanpa jendela 30 bln', no30), ('A+B', SET_AB), ('semua (A+B+C)', FULL)]:
    for kind in ['LR', 'HGB']:
        f, fs, r_, rs = cv_score(cols, kind); rows.append(dict(set=nm, n_fitur=len(cols), model=kind, macro_F1=f, sd_F1=fs, recall_High=r_, sd_recall=rs))
TIER = pd.DataFrame(rows); TIER.to_csv(OUT / 'tier_comparison_cv.csv', index=False)
display(TIER.pivot(index=['set', 'n_fitur'], columns='model', values=['macro_F1', 'sd_F1', 'recall_High']).round(3))
yd = (DEVSET.risk_class == 'High').astype(int)
print('AUC High (train+valid) per jendela timing:', {c: round(roc_auc_score(yd[DEVSET[c].notna()], DEVSET[c].dropna()), 3) for c in ['pay_late_6m', 'pay_late_12m', 'pay_late_30m', 'tax_late_6m', 'tax_late_12m', 'tax_late_30m']})
SET_CORE = SET_AB
print(f'→ Set awal untuk notebook 04 = A+B ({len(SET_CORE)} fitur). Fitur tingkat C tetap ada di file dan boleh diuji ulang saat HPO.')

### E3 · Satu kali cek train → valid (referensi untuk HPO)
Ini angka acuan untuk notebook 05: model dengan setelan default, dilatih di **train** dan dinilai di **valid**. HPO nanti harus memperbaikinya tanpa pernah melihat test.

In [ ]:
TR, VA = R[R.split == 'train'], R[R.split == 'valid']; ytr = TR.risk_class.map({'Low': 0, 'Medium': 1, 'High': 2}).values; yva = VA.risk_class.map({'Low': 0, 'Medium': 1, 'High': 2}).values
rows = []
for nm, cols in [('set inti A+B', SET_CORE), ('set lengkap', FULL), ('set portabel', PORTABLE)]:
    for kind in ['LR', 'HGB']:
        p_ = make_model(cols, kind).fit(TR, ytr).predict(VA)
        rows.append(dict(set=nm, model=kind, macro_F1_valid=f1_score(yva, p_, average='macro'), recall_High_valid=recall_score(yva, p_, labels=[2], average='macro')))
REF = pd.DataFrame(rows); display(REF.round(3)); REF.to_csv(OUT / 'reference_train_valid.csv', index=False)

### E4 · 🔎 Cek fitur anomali online terhadap ground truth (evaluasi saja; tanpa perusahaan test)
Pertanyaannya: apakah fitur **online** masih menangkap kejadian sebaik skor **retrospektif** di EDA (F10g)? Untuk menjaga test tetap bersih, cek ini hanya memakai nasabah di train, valid, dan unlabeled. Skor per nasabah = nilai paling ekstrem sepanjang 2024-07 s.d. 2026-06. Dihitung juga **lead time** distress (berapa bulan sebelum mulainya distress fitur timing pertama kali melewati ambang) beserta **biayanya**: berapa persen nasabah normal yang pernah memicu alarm yang sama.

In [ ]:
gta = rd('gt_anomaly_events')
ok_ids = PNL.loc[PNL.split != 'test', 'company_id'].unique()
P2 = PNL[(PNL.ym >= '2024-07') & PNL.company_id.isin(ok_ids)]
agg = P2.groupby('company_id').agg(spike=('inflow_resid_z', 'max'), chg_down=('chg_3v6', lambda s_: -s_.min()), chg_up=('chg_3v6', 'max'),
                                   cusum_down=('cusum_down', 'max'), cusum_up=('cusum_up', 'max'), buyer_exit=('buyer_exit_share', 'max'),
                                   anchor=('exp_shocked_anchor_t', 'max'), timing=('timing_delta_t', 'max'), out_in=('out_in_delta', 'max'), capture_down=('capture_chg_6m', lambda s_: -s_.min()))
SIGNAL_LBL = {'spike': 'z residual maks (online)', 'chg_down': 'chg_3v6 turun', 'chg_up': 'chg_3v6 naik', 'cusum_down': 'CUSUM turun', 'cusum_up': 'CUSUM naik',
              'buyer_exit': 'porsi pembeli berhenti', 'anchor': 'paparan anchor (online)', 'timing': 'Δ timing bayar', 'out_in': 'Δ rasio outflow/inflow', 'capture_down': 'capture turun 6 bln'}
EV = ['spike', 'expansion', 'distress', 'contagion', 'buyer_exit', 'wallet_erosion']
EVL = {'spike': 'lonjakan sesaat', 'expansion': 'ekspansi', 'distress': 'distress (+ sinyal dini)', 'contagion': 'efek berantai', 'buyer_exit': 'pembeli utama keluar', 'wallet_erosion': 'erosi wallet'}
g_ok = gta[gta.company_id.isin(agg.index)]
MA = pd.DataFrame({EVL[e]: [roc_auc_score(agg.index.isin(g_ok.loc[g_ok.event_type == e, 'company_id']).astype(int), agg[s_].fillna(agg[s_].min())) for s_ in agg] for e in EV},
                  index=[SIGNAL_LBL[s_] for s_ in agg])
fig, ax = plt.subplots(figsize=(11.5, 5))
im = ax.imshow(MA.values, cmap='Blues', vmin=.5, vmax=1, aspect='auto'); ax.grid(False)
ax.set_xticks(range(len(EV))); ax.set_xticklabels(MA.columns, fontsize=8); ax.set_yticks(range(len(MA))); ax.set_yticklabels(MA.index, fontsize=8)
for i in range(MA.shape[0]):
    for j in range(MA.shape[1]):
        v = MA.values[i, j]; ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='white' if v > .8 else C['ink'], fontweight='bold' if v == MA.values[:, j].max() else 'normal')
plt.colorbar(im, ax=ax, shrink=.8, label='AUC'); ax.set_title('E4 · Fitur anomali ONLINE × jenis kejadian (ground truth; tanpa perusahaan test)')
plt.tight_layout(); save_fig(fig, 'E4_anomaly_online_matrix'); plt.show()
eda_m = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_eda' / 'anomaly_signal_event_auc.csv', index_col=0) if (PROJECT_ROOT / 'outputs' / '02_eda' / 'anomaly_signal_event_auc.csv').exists() else None
cmp_ = pd.DataFrame({'AUC terbaik online (E4)': MA.max(), 'sinyal online terbaik': MA.idxmax()})
if eda_m is not None: cmp_['AUC terbaik retrospektif (EDA F10g)'] = eda_m.max().reindex(cmp_.index)
display(cmp_.round(3))
dis_ = g_ok[g_ok.event_type == 'distress'][['company_id', 'start_month']]
non_ev = [c for c in ok_ids if c not in set(gta.company_id)]
rows = []
for feat, thr in [('timing_delta_t', .25), ('timing_delta_t', .3), ('timing_jump_3v9', .25), ('timing_jump_3v9', .3)]:
    W = PNL[PNL.company_id.isin(ok_ids)].pivot(index='company_id', columns='ym', values=feat)
    leads = []
    for r_ in dis_.itertuples():
        hit = W.loc[r_.company_id][W.loc[r_.company_id] > thr]
        if len(hit): leads.append((pd.Period(r_.start_month, 'M') - pd.Period(hit.index[0], 'M')).n)
    leads = np.array(leads)
    rows.append(dict(fitur=feat, ambang=thr, lead_median_bln=np.median(leads), terdeteksi_ge_1bln_awal=np.mean(leads >= 1), terdeteksi_ge_2bln_awal=np.mean(leads >= 2),
                     nasabah_normal_pernah_alarm=(W.reindex(non_ev).loc[:, '2024-07':] > thr).any(axis=1).mean()))
LEAD = pd.DataFrame(rows); display(LEAD.round(3)); LEAD.to_csv(OUT / 'distress_lead_time_tradeoff.csv', index=False)
print('→ Trade-off: jendela pendek memberi peringatan ±2 bulan lebih awal, tetapi alarm palsunya jauh lebih banyak. Keduanya disimpan;')
print('  model notebook 04 yang menggabungkan (bersama CUSUM & arus kas) untuk menekan alarm palsu.')
print('Catatan trivialitas: AUC paparan anchor ≈ 1 untuk "efek berantai" karena kejadian itu di dunia sintetis DIDEFINISIKAN dari ketergantungan ke anchor yang')
print('bermasalah. Ini bukti detektor anchor online bekerja, bukan bukti kehebatan model. Cukup aturan transparan (Wu & Keogh, 2021).')
MA.to_csv(OUT / 'anomaly_online_signal_event_auc.csv')

## F. Simpan dataset final
Semua yang dibutuhkan notebook 04–07 dan dashboard ditulis ke `data/processed/`:

| Berkas | Isi |
|---|---|
| `features_risk` (.parquet / .csv.gz) | 2.000 baris: kunci (`company_id`, `split`, `cv_fold`, `customer_type`, `industry`, `size_class_reference`), target `risk_class`, semua fitur |
| `features_anomaly_panel` | 42.000 baris nasabah × bulan: kunci + fitur online |
| `feature_sets.json` | daftar kolom per kelompok; set **inti (A+B)**, **lengkap**, **portabel**, **anomali**; kolom mentah untuk `PeerZScaler` |
| `feature_dictionary.csv` | definisi, kelompok, sumber temuan EDA, tingkat bukti, % kosong, AUC High di train |
| `split.csv` | `split` (train/valid/test/unlabeled) + `cv_fold` |
| `eval_targets/` | ground truth yang dirapikan: **hanya untuk notebook 06** |

In [ ]:
KEYS = ['company_id', 'split', 'cv_fold', 'customer_type', 'industry', 'size_class_reference', 'risk_class']
feat_cols = list(dict.fromkeys(FULL + [c for c in DICT_DF.fitur if c in R and c not in FULL]))
OUT_RISK = R.reset_index()[KEYS + feat_cols]
OUT_PNL = PNL[['company_id', 'ym', 'split', 'cv_fold', 'industry', 'inflow_log', 'inflow_resid'] + [c for c in ANOM_COLS if c not in ('inflow_log', 'inflow_resid')]]
f1_ = write_table(OUT_RISK, 'features_risk'); f2_ = write_table(OUT_PNL, 'features_anomaly_panel')

DICT_ALL = pd.DataFrame(DICT).drop_duplicates('fitur')
trn = R[R.split == 'train']; yH = (trn.risk_class == 'High').astype(int)
def auc_or_nan(c):
    if c not in trn or trn[c].notna().sum() < 50 or trn[c].nunique() < 2: return np.nan
    m = trn[c].notna(); a = roc_auc_score(yH[m], trn.loc[m, c]); return max(a, 1 - a)
DICT_ALL['pct_kosong'] = [round(100 * (R[c].isna().mean() if c in R else PNL[c].isna().mean()), 1) for c in DICT_ALL.fitur]
DICT_ALL['AUC_High_train'] = [round(auc_or_nan(c), 3) if c in R else np.nan for c in DICT_ALL.fitur]
DICT_ALL['di_set_inti'] = DICT_ALL.fitur.isin(SET_CORE)
DICT_ALL['tabel'] = np.where(DICT_ALL.kelompok == 'anomali (panel)', 'features_anomaly_panel', 'features_risk')
DICT_ALL.to_csv(PROC / 'feature_dictionary.csv', index=False)
SETS = {'groups': GROUPS, 'set_core': SET_CORE, 'set_client': FULL, 'set_portable': PORTABLE,
        'peer_raw': RATIO_COLS + PEER_KEYS, 'peer_z': PZ_OUT, 'anomaly': [c for c in ANOM_COLS], 'target': 'risk_class', 'split_col': 'split', 'fold_col': 'cv_fold',
        'notes': {'risk_model': 'fit di split==train (atau CV cv_fold di train+valid); nilai sekali di test',
                  'peer_z': 'kolom *_pz di file di-fit pada train+unlabeled; untuk CV gunakan PeerZScaler di pipeline',
                  'anomaly': 'semua fitur online (data ≤ bulan t); mulai pakai dari 2024-07; jangan pakai relationship_end',
                  'prospect': 'pakai set_portable; transaksi prospect hanya terlihat sebagian'}}
(PROC / 'feature_sets.json').write_text(json.dumps(SETS, ensure_ascii=False, indent=1), encoding='utf-8')

# --- target evaluasi (ground truth) — terpisah dari fitur, hanya untuk notebook 06 ---
rd('gt_prospect_risk').to_csv(EVAL / 'prospect_true_risk.csv', index=False)
rd('gt_client_risk').to_csv(EVAL / 'client_true_risk_and_label_flip.csv', index=False)
ev_rows = []
for r_ in gta.itertuples():
    for m_ in pd.period_range(r_.start_month, r_.end_month, freq='M'):
        ev_rows.append((r_.company_id, str(m_), r_.event_type, r_.direction, int(str(m_) == r_.start_month)))
pd.DataFrame(ev_rows, columns=['company_id', 'ym', 'event_type', 'direction', 'is_onset']).to_csv(EVAL / 'anomaly_events_monthly.csv', index=False)
print('data/processed:', sorted(p_.name for p_ in PROC.iterdir() if p_.is_file()))
print('eval_targets  :', sorted(p_.name for p_ in EVAL.iterdir()))
print(f'features_risk {OUT_RISK.shape} · features_anomaly_panel {OUT_PNL.shape} · {len(DICT_ALL)} fitur di kamus')
display(DICT_ALL.groupby(['tabel', 'tingkat_bukti']).size().unstack(fill_value=0))

**Contoh pemakaian di notebook 04/05** (tanpa join CSV lagi):
```python
import sys; sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from features import load_features, load_feature_sets, PeerZScaler
df   = load_features(PROJECT_ROOT, 'risk');  fs = load_feature_sets(PROJECT_ROOT)
X_tr = df[df.split == 'train'][fs['set_core']];  y_tr = df.loc[df.split == 'train', 'risk_class']
X_va = df[df.split == 'valid'][fs['set_core']]                  # untuk HPO; test hanya sekali di akhir
panel = load_features(PROJECT_ROOT, 'anomaly')                  # alarm bulanan
```

In [ ]:
summary = f"""
| Bagian | Hasil |
|---|---|
| Split | train {int((R.split == 'train').sum())} / valid {int((R.split == 'valid').sum())} / test {int((R.split == 'test').sum())} (= holdout NB02) / unlabeled {int((R.split == 'unlabeled').sum())}; `cv_fold` 0–4 di train+valid |
| Fitur risiko | {len(FULL)} fitur ({len(PORTABLE)} portabel); set inti A+B = {len(SET_CORE)} fitur |
| Fitur anomali | {len(ANOM_COLS)} fitur online × 42.000 nasabah-bulan |
| Ablation | kelompok paling berharga (leave-one-out): timing bayar & konsentrasi; tingkat C (jaringan/geografi/kota) tidak menambah, bahkan menurunkan |
| Set inti A+B (CV 2×5) | macro-F1 {TIER.query("set == 'A+B' and model == 'HGB'").macro_F1.iloc[0]:.3f} (HGB) / {TIER.query("set == 'A+B' and model == 'LR'").macro_F1.iloc[0]:.3f} (LR), ±0,03; recall High {TIER.query("set == 'A+B' and model == 'LR'").recall_High.iloc[0]:.2f} (LR) |
| Train → valid (default) | macro-F1 {REF.query("set == 'set inti A+B' and model == 'LR'").macro_F1_valid.iloc[0]:.3f} (LR, set inti) — patokan HPO |
| Jendela timing | 30 bln > 12 bln > 6 bln (AUC 0,85–0,86 vs 0,80 vs 0,73–0,75) |
| Anomali online (🔎, tanpa test) | AUC terbaik per jenis (tingkat perusahaan): lonjakan {MA.max()['lonjakan sesaat']:.2f} · ekspansi {MA.max()['ekspansi']:.2f} · distress {MA.max()['distress (+ sinyal dini)']:.2f} · efek berantai {MA.max()['efek berantai']:.2f} (trivial, sesuai definisi) · buyer exit {MA.max()['pembeli utama keluar']:.2f} · erosi {MA.max()['erosi wallet']:.2f} |
| Lead time distress | jendela cepat 3v9: median {LEAD.query("fitur == 'timing_jump_3v9' and ambang == 0.3").lead_median_bln.iloc[0]:.0f} bln lebih awal, tapi alarm palsu {LEAD.query("fitur == 'timing_jump_3v9' and ambang == 0.3").nasabah_normal_pernah_alarm.iloc[0]:.0%} |
"""
display(Markdown(summary))

**Catatan untuk notebook 04–05**
1. **Set awal = inti A+B.** Menambah semua fitur tidak membantu. Fitur tingkat C boleh diuji ulang saat HPO, tapi keputusan tetap di valid/CV, bukan di test.
2. **Trade-off class weight.** `class_weight='balanced'` menaikkan recall High, tapi bisa menurunkan macro-F1. Notebook 05 membandingkan class weight, cost-sensitive, dan SMOTE, lalu memilih ambang keputusan di valid.
3. **Selisih dengan probe EDA (0,72).** Probe dihitung dengan statistik peer dari seluruh non-holdout dan satu kali CV. Angka di notebook ini lebih jujur, karena peer di-fit per fold dan CV diulang, dan selisihnya masih di dalam ±1 simpangan baku.
4. **Anomali.** Lonjakan, buyer exit, efek berantai, dan erosi cukup dengan aturan transparan. Di tingkat perusahaan, distress juga hampir sempurna terdeteksi oleh Δ timing. Tantangannya ada pada **waktu**: jendela cepat memberi peringatan ±2 bulan lebih awal, tapi 45–65% nasabah normal pernah ikut memicu alarm. Jadi ML untuk anomali difokuskan pada (a) **distress per bulan**, untuk menyeimbangkan lead time dan alarm palsu dengan menggabungkan timing dua jendela, CUSUM, dan arus kas; dan (b) **ekspansi**, sinyal yang paling lemah (AUC ±0,75). Metrik utamanya: presisi top-k alarm per bulan dan lead time.

## Daftar pustaka
Daftar yang sama disimpan di `outputs/03_fe/references.csv` dan `REFERENCES.md`. Status ✔ = metadata dicocokkan ke Crossref / situs penerbit (10–11 Okt 2026).

In [ ]:
REFS = [  # (kunci, rujukan, doi/url, bagian, untuk apa, status)
    ('Varma2006', 'Varma, S. & Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. BMC Bioinformatics, 7, 91.', '10.1186/1471-2105-7-91', 'B', 'alasan memisahkan valid/CV (HPO) dari test', '✔'),
    ('Cawley2010', 'Cawley, G. C. & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. JMLR, 11, 2079–2107.', 'https://jmlr.org/papers/v11/cawley10a.html', 'B', 'bias seleksi model bila test ikut dipakai', '✔'),
    ('Kapoor2023', 'Kapoor, S. & Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. Patterns, 4(9), 100804.', '10.1016/j.patter.2023.100804', 'B, C', 'fit statistik hanya di train; pipeline per fold', '✔'),
    ('Leys2013', 'Leys, C. dkk. (2013). Detecting outliers: Do not use standard deviation around the mean, use absolute deviation around the median. JESP, 49(4).', '10.1016/j.jesp.2013.03.013', 'C1, D', 'z robust median/MAD (peer & panel)', '✔'),
    ('Sperrin2020', 'Sperrin, M. dkk. (2020). Missing data should be handled differently for prediction than for description or causal explanation. J. Clin. Epidemiol., 125.', '10.1016/j.jclinepi.2020.03.028', 'C1', 'has_report & indikator kosong sebagai fitur', '✔'),
    ('Bell1983', 'Bell, W. R. & Hillmer, S. C. (1983). Modeling time series with calendar variation. JASA, 78(383).', '10.1080/01621459.1983.10478005', 'C3, D', 'koreksi musim industri × bulan (Lebaran bergerak)', '✔'),
    ('Du2007', 'Du, R. Y., Kamakura, W. A. & Mela, C. F. (2007). Size and share of customer wallet. Journal of Marketing, 71(2).', '10.1509/jmkg.71.2.094', 'C3, D', 'capture ratio sebagai proxy share of wallet', '✔'),
    ('Bernal2017', 'Bernal, J. L., Cummins, S. & Gasparrini, A. (2017). Interrupted time series regression for the evaluation of public health interventions: a tutorial. IJE, 46(1).', '10.1093/ije/dyw098', 'C6, D', 'detektor patahan tren anchor', '✔'),
    ('Battiston2012', 'Battiston, S. dkk. (2012). DebtRank: Too central to fail? Scientific Reports, 2, 541.', '10.1038/srep00541', 'C6, D', 'paparan ke anchor bermasalah', '✔'),
    ('Page1954', 'Page, E. S. (1954). Continuous inspection schemes. Biometrika, 41(1/2), 100–115.', '10.1093/biomet/41.1-2.100', 'D', 'CUSUM online', '✔'),
    ('Chandola2009', 'Chandola, V., Banerjee, A. & Kumar, V. (2009). Anomaly detection: A survey. ACM Computing Surveys, 41(3).', '10.1145/1541880.1541882', 'D', 'anomali titik vs kolektif vs kontekstual', '✔'),
    ('Wu2021', 'Wu, R. & Keogh, E. J. (2021). Current time series anomaly detection benchmarks are flawed and are creating the illusion of progress. IEEE TKDE.', '10.1109/TKDE.2021.3112126', 'E4', 'trivialitas: kapan aturan sederhana cukup', '✔'),
    ('Shimodaira2000', 'Shimodaira, H. (2000). Improving predictive inference under covariate shift by weighting the log-likelihood function. JSPI, 90(2).', '10.1016/S0378-3758(00)00115-4', 'E2', 'set portabel + importance weighting untuk prospect', '✔'),
    ('Pargent2022', 'Pargent, F., Pfisterer, F., Thomas, J. & Bischl, B. (2022). Regularized target encoding outperforms traditional methods in supervised machine learning with high cardinality features. Computational Statistics, 37.', '10.1007/s00180-022-01207-6', 'C7', 'alasan mengganti one-hot 40 kecamatan dengan atribut numerik', '○'),
]
refs = pd.DataFrame(REFS, columns=['kunci', 'rujukan', 'doi_atau_url', 'bagian', 'untuk_apa', 'status'])
refs['tautan'] = np.where(refs.doi_atau_url.str.startswith('10.'), 'https://doi.org/' + refs.doi_atau_url, refs.doi_atau_url)
refs.to_csv(OUT / 'references.csv', index=False)
with open(OUT / 'REFERENCES.md', 'w', encoding='utf-8') as fh:
    fh.write('# Rujukan — Notebook 03 Feature Engineering\n\nStatus: ✔ = dicocokkan ke Crossref / situs penerbit · ○ = belum dicek.\n\n')
    fh.write('| Kunci | Bagian | Untuk apa | Rujukan | Tautan | Status |\n|---|---|---|---|---|---|\n')
    for r_ in refs.itertuples(): fh.write(f'| {r_.kunci} | {r_.bagian} | {r_.untuk_apa} | {r_.rujukan} | {r_.tautan} | {r_.status} |\n')
print(f"{len(refs)} rujukan · {(refs.status == '✔').sum()} terverifikasi")
display(refs[['kunci', 'bagian', 'untuk_apa', 'tautan', 'status']])